In [1]:
import csv

def read_weather_data(file_path):
    year_temp = {}

    with open(file_path, 'r') as f:
        reader = csv.DictReader(f)

        for row in reader:
            # You may need to adjust these field names to match your actual CSV
            date = row['Formatted Date']
            temperature = float(row['Temperature (C)'])

            # Extract year from the date string, assuming format like "2016-01-01 00:00:00"
            year = int(date[:4])

            if year not in year_temp:
                year_temp[year] = []

            year_temp[year].append(temperature)

    return year_temp

def find_extremes(year_temp):
    result = {}

    for year, temps in year_temp.items():
        result[year] = {
            'max': max(temps),
            'min': min(temps)
        }

    hottest = max(result, key=lambda y: result[y]['max'])
    coolest = min(result, key=lambda y: result[y]['min'])

    return hottest, result[hottest], coolest, result[coolest]

if __name__ == "__main__":
    data = read_weather_data('weatherHistory.csv')
    hottest_year, hottest_data, coolest_year, coolest_data = find_extremes(data)

    print(f"Hottest Year: {hottest_year} with temperature {hottest_data['max']}°C")
    print(f"Coolest Year: {coolest_year} with temperature {coolest_data['min']}°C")


Hottest Year: 2007 with temperature 39.90555555555555°C
Coolest Year: 2012 with temperature -21.822222222222223°C


In [3]:
import csv
from collections import defaultdict

# ---------- MAPPER FUNCTION ----------
def mapper(file_path):
    year_temp_pairs = []

    with open(file_path, 'r') as f:
        reader = csv.DictReader(f)

        for row in reader:
            try:
                date = row['Formatted Date']
                temperature = float(row['Temperature (C)'])
                year = int(date[:4])
                year_temp_pairs.append((year, temperature))
            except (KeyError, ValueError):
                # Skip rows with missing or bad data
                continue

    return year_temp_pairs


# ---------- REDUCER FUNCTION ----------
def reducer(year_temp_pairs):
    year_to_temps = defaultdict(list)

    # Group temperatures by year
    for year, temp in year_temp_pairs:
        year_to_temps[year].append(temp)

    year_extremes = {}
    for year, temps in year_to_temps.items():
        year_extremes[year] = {
            'max': max(temps),
            'min': min(temps)
        }

    return year_extremes


# ---------- FIND HOTTEST & COOLEST YEARS ----------
def find_extremes(year_extremes):
    hottest = max(year_extremes, key=lambda y: year_extremes[y]['max'])
    coolest = min(year_extremes, key=lambda y: year_extremes[y]['min'])

    return hottest, year_extremes[hottest], coolest, year_extremes[coolest]


# ---------- MAIN FUNCTION ----------
if __name__ == "__main__":
    year_temp_pairs = mapper('weatherHistory.csv')
    year_extremes = reducer(year_temp_pairs)
    hottest_year, hottest_data, coolest_year, coolest_data = find_extremes(year_extremes)

    print(f"Hottest Year: {hottest_year} with temperature {hottest_data['max']}°C")
    print(f"Coolest Year: {coolest_year} with temperature {coolest_data['min']}°C")


Hottest Year: 2007 with temperature 39.90555555555555°C
Coolest Year: 2012 with temperature -21.822222222222223°C
